# P3 · Selección de la configuración de agrupamiento

Este notebook documenta **cómo se eligió** la configuración que usa el experimento base de P3
(`mibici/modeling/p3.py`): KMeans con k=3 sobre las estaciones con al menos 180 días con
eventos en entrenamiento. Cada etapa llama funciones del módulo; aquí solo se orquesta la
secuencia y se explica cada decisión.

| Etapa | Pregunta | Funciones |
|---|---|---|
| 1. Revisión | ¿Los meses de entrenamiento están íntegros y qué estaciones son elegibles? | `dataset.cargar_entrenamiento_revision`, `features.preparar_perfiles_p3`, `dataset.exportar_revision` |
| 2. Comparación de k | ¿Qué método y número de grupos describen mejor los perfiles? | `features.perfiles_elegibles_train`, `evaluation.comparar`, `evaluation.exportar_comparacion` |
| 3. Sensibilidad a la historia | ¿Cambian los grupos si se exige más historia (180 o 300 días)? | `evaluation.sensibilidad_historia`, `evaluation.exportar_sensibilidad` |
| 4. Validación en agosto | ¿Los grupos de entrenamiento se mantienen en agosto? | `evaluation.modelo_tentativo`, `grupo_domingo`, `recurrencia_grupo_domingo`, `validacion_agosto`, `resumen_validacion`, `exportar_validacion` |

**Salvaguardas.** Ninguna etapa lee septiembre (prueba). Las etapas 1 a 3 solo usan
entrenamiento (agosto de 2025 a julio de 2026); la 4 lee agosto de 2026 asignándolo a
centroides de entrenamiento, sin reajustar. Cada etapa escribe en una carpeta **nueva** y
se detiene si ya existe.

**Nueva exploración frente a evidencia histórica.** Este notebook genera una exploración
nueva en `reports/p3-exploracion/<ETIQUETA>/`. Las evidencias originales de la selección
(`reports/p3-*-20261018/`, rutas en `mibici/config.py`) no se leen ni se modifican aquí; la
auditoría que las usa está en `3.0-fvep-p3-auditoria.ipynb`.

Requisito: haber corrido la ingesta (`python -m mibici.dataset`).

In [ ]:
import json

import pandas as pd

from mibici.config import MANIFESTS_DIR, REPORTS_DIR
from mibici.dataset import cargar_entrenamiento_revision, cargar_estaciones, exportar_revision
from mibici.evaluation import (
    comparar,
    exportar_comparacion,
    exportar_sensibilidad,
    exportar_validacion,
    grupo_domingo,
    modelo_tentativo,
    recurrencia_grupo_domingo,
    resumen_validacion,
    sensibilidad_historia,
    validacion_agosto,
)
from mibici.features import perfiles_elegibles_train, preparar_perfiles_p3

Elijan una etiqueta nueva para cada exploración (por ejemplo, la fecha). Las carpetas de salida no deben existir.

In [ ]:
ETIQUETA = "AAAAMMDD"  # cambiar antes de ejecutar
SALIDA = REPORTS_DIR / "p3-exploracion" / ETIQUETA
REVISION = SALIDA / "revision"
COMPARACION = SALIDA / "comparacion"
HISTORIA = SALIDA / "historia"
VALIDACION = SALIDA / "validacion-agosto"
SALIDA

## 1. Revisión de la preparación

Se leen los 12 meses de entrenamiento verificando SHA-256 y número de registros contra
`manifests/mibici_processed.json`, y se construye un perfil por estación: 168 proporciones
de salidas y 168 de llegadas (día de la semana × hora). La elegibilidad es provisional:
28 días con eventos, 100 movimientos, eventos en los siete días y ambas direcciones.
Los ceros indican ausencia de registros, no demanda cero confirmada.

In [ ]:
viajes, cobertura, manifiesto = cargar_entrenamiento_revision()
perfiles = preparar_perfiles_p3(viajes, cargar_estaciones())
del viajes
exportar_revision(REVISION, perfiles, cobertura, manifiesto)

## 2. Comparación de configuraciones

Se comparan KMeans y Ward con k de 2 a 6 sobre las estaciones elegibles, con distancia
euclidiana sobre `sqrt(p)/sqrt(2)` (salidas y llegadas pesan igual). Las etiquetas no
implican un ranking: la tabla resume separación (silhouette) y tamaños de grupo.

In [ ]:
entrada = REVISION / "perfiles_entrenamiento.parquet"
elegibles = perfiles_elegibles_train(pd.read_parquet(entrada))
tabla, etiquetas, representantes = comparar(elegibles)
exportar_comparacion(COMPARACION, entrada, elegibles, tabla, etiquetas, representantes)

## 3. Sensibilidad a la historia mínima

Se repite la comparación exigiendo al menos 180 y 300 días con eventos (cohortes
predeclaradas en `config.P3_COHORTES`) y se mide cuánto cambian los grupos respecto a la
comparación anterior (ARI) y cuánto se asocian con la antigüedad de la estación (η²).
De aquí sale la cohorte de 180 días.

In [ ]:
resultado = sensibilidad_historia(
    pd.read_parquet(entrada), pd.read_csv(COMPARACION / "asignaciones.csv")
)
exportar_sensibilidad(HISTORIA, entrada, COMPARACION, resultado)
resultado["conteos"]

## 4. Validación tentativa en agosto

**Requisito: protocolo previo escrito a mano.** Antes de ejecutar esta sección hay que crear
`VALIDACION/PROTOCOLO_PREVIO.txt` con lo que se va a comprobar y los criterios para
interpretarlo, **antes de mirar agosto**. El código no genera ese archivo: las funciones se
detienen con `Requiere protocolo previo y salida no ejecutada` si falta o si la carpeta ya
tiene un `resumen.json`. Las funciones que leen agosto vuelven a verificarlo.

Se reajusta en entrenamiento el KMeans tentativo (k=3, cohorte de 180 días) y se comprueba
que reproduce la comparación anterior. Después se describe el grupo con más uso en
domingo (solo con entrenamiento) y se asigna agosto a los centroides de entrenamiento,
sin reajustarlos. Septiembre no se lee.

In [ ]:
manifiesto = json.loads((MANIFESTS_DIR / "mibici_processed.json").read_text(encoding="utf-8"))
registros = {r["filename"]: r for r in manifiesto["files"]}

train, x, modelo, centros = modelo_tentativo(pd.read_parquet(entrada), HISTORIA, VALIDACION)
grupo_dom, miembros = grupo_domingo(train, modelo)
recurrencia = recurrencia_grupo_domingo(miembros, registros)
validacion = validacion_agosto(
    modelo, train, x, recurrencia["julio"], registros, recurrencia["integridad"], VALIDACION
)
resumen = resumen_validacion(train, modelo, grupo_dom, miembros, recurrencia["julio"], validacion)
exportar_validacion(VALIDACION, train, modelo, centros, miembros, recurrencia, validacion, resumen)

## Resultado de la selección

Con esta secuencia se fijó la configuración de `references/p3_frozen.json`: KMeans k=3,
cohorte de ≥180 días, semilla 42, `n_init` 20, `max_iter` 300, algoritmo de Lloyd. Esa
decisión se tomó antes de evaluar septiembre. El experimento base
(`python -m mibici.modeling.train --pregunta P3`) reajusta esa misma configuración y marca
`reproduce_configuracion_congelada=true` cuando sus métricas coinciden con las congeladas.